## Mistral 7B v0.2

In [2]:
from transformers import AutoTokenizer, MistralForCausalLM

trained_weights = "mistralai/Mistral-7B-Instruct-v0.2"

model = MistralForCausalLM.from_pretrained(trained_weights)
tokenizer = AutoTokenizer.from_pretrained(trained_weights)

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [3]:
# https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.2 
# https://mistral.ai/news/announcing-mistral-7b

prompt = "Hey, what is a finance bro?"
inputs = tokenizer(prompt, return_tensors="pt")

generate_ids = model.generate(inputs.input_ids, max_length = 30)
tokenizer.batch_decode(generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


'Hey, what is a finance bro?\n\nA finance broker is a professional who helps individuals and businesses secure financing for various purposes, such as'

In [5]:
prompt = " Classify this phrase as positive, negative or neutral: The food was good, but not amazing."
inputs = tokenizer(prompt, return_tensors="pt")

generate_ids = model.generate(inputs.input_ids, max_length = 100)
tokenizer.batch_decode(generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


'Classify this phrase as positive, negative or neutral: The food was good, but not amazing.\n\nAnswer: Neutral. The phrase acknowledges that the food was satisfactory, but it did not elicit a strong positive response.'

In [6]:
prompt = " Classify the market reaction to this phrase as positive, negative or neutral: US has hit more than 1,700 targets in Iran so far, officials say."
inputs = tokenizer(prompt, return_tensors="pt")

generate_ids = model.generate(inputs.input_ids, max_length = 100)
tokenizer.batch_decode(generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


'Classify the market reaction to this phrase as positive, negative or neutral: US has hit more than 1,700 targets in Iran so far, officials say.\n\nNeutral: The statement is factual and does not contain any positive or negative connotation. It simply reports the number of targets hit by the US in Iran, without expressing an opinion or making a judgment. The market reaction to this statement would likely be neutral as well, as it does not provide'

In [9]:
import gc
import torch

# # Example: you used these
# # model, tokenizer, optimizer, lr_scheduler, train_dataloader, etc.

del model
del tokenizer  # if on GPU or large
# del optimizer
# del lr_scheduler
# del train_dataloader   # if it holds GPU tensors, optional

gc.collect()              # force Python garbage collection
torch.cuda.empty_cache()  # release unused cached VRAM


NameError: name 'model' is not defined

In [10]:
gc.collect()              # force Python garbage collection
torch.cuda.empty_cache()

# LoRA (Low-Rank Adaptation) fine-tuning

- https://huggingface.co/learn/llm-course/chapter11/4
- https://huggingface.co/mistralai/Mistral-7B-v0.1
- https://huggingface.co/mistralai/Mistral-7B-v0.3
- https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.2 <--

## Dataset Prep

In [5]:
from datasets import load_dataset

ds = load_dataset("virattt/financial-qa-10K")

In [6]:
df2 = ds["train"].to_pandas()
df2.head()

,question,answer,context,ticker,filing
0,What area did NVIDIA initially focus on before...,NVIDIA initially focused on PC graphics.,"Since our original focus on PC graphics, we ha...",NVDA,2023_10K
1,What are some of the recent applications of GP...,Recent applications of GPU-powered deep learni...,Some of the most recent applications of GPU-po...,NVDA,2023_10K
2,What significant invention did NVIDIA create i...,NVIDIA invented the GPU in 1999.,Our invention of the GPU in 1999 defined moder...,NVDA,2023_10K
3,How does NVIDIA's platform strategy contribute...,NVIDIA's platform strategy brings together har...,"NVIDIA has a platform strategy, bringing toget...",NVDA,2023_10K
4,What does NVIDIA's CUDA programming model enable?,NVIDIA's CUDA programming model opened the par...,With our introduction of the CUDA programming ...,NVDA,2023_10K


In [7]:
# FORMAT 1: Teaches the model to use a context to answer (using RAG later)
format_1 = """Use the following context to answer the question.

Context: {context}

Question: {question}
"""

# FORMAT 2 : When no context is available
format_2 = """ {question} """


In [8]:
# example
question = df2.question[0]
context = df2.context[0]
answer = df2.answer[0]


In [9]:
# create dataset to perform the training
# using the correct format
# https://huggingface.co/docs/transformers/main/chat_templating
from transformers import AutoTokenizer, MistralForCausalLM

trained_weights = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(trained_weights)


In [10]:
content = format_1.format(context = context, question=question)
print(content)

chat = [{"role": "user", "content": content},
        {"role": "assistant", "content": answer}]
tokenizer.apply_chat_template(chat, tokenize = False)

Use the following context to answer the question.

Context: Since our original focus on PC graphics, we have expanded to several other large and important computationally intensive fields.

Question: What area did NVIDIA initially focus on before expanding to other computationally intensive fields?



'<s> [INST] Use the following context to answer the question.\n\nContext: Since our original focus on PC graphics, we have expanded to several other large and important computationally intensive fields.\n\nQuestion: What area did NVIDIA initially focus on before expanding to other computationally intensive fields?\n [/INST] NVIDIA initially focused on PC graphics.</s>'

In [11]:
def format_row(row, tokenizer, include_context = True):

    format_1 = """Use the following context to answer the question.

    Context: {context}

    Question: {question}
    """

    try: 
        context = row["context"]
        question = row["question"]
        answer = row["answer"]

        # FORMAT 1: Teaches the model to use a context to answer (using RAG later)
        if include_context:
            content = format_1.format(context = context, question=question)

            chat = [{"role": "user", "content": content},
            {"role": "assistant", "content": answer}]
        
        # FORMAT 2 : When no context is available: use only the question
        else:
            chat = [{"role": "user", "content": question},
            {"role": "assistant", "content": answer}]

        # format chat
    
        chat = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=False)
    
        return chat

    except Exception as e: 
        print(e)
        return None    




In [12]:
#  test 1
chat = format_row(df2.iloc[0], tokenizer)
print(chat)

# test 2
chat = format_row(df2.iloc[0], tokenizer, include_context=False)
print(chat)


<s> [INST] Use the following context to answer the question.

    Context: Since our original focus on PC graphics, we have expanded to several other large and important computationally intensive fields.

    Question: What area did NVIDIA initially focus on before expanding to other computationally intensive fields?
     [/INST] NVIDIA initially focused on PC graphics.</s>
<s> [INST] What area did NVIDIA initially focus on before expanding to other computationally intensive fields? [/INST] NVIDIA initially focused on PC graphics.</s>


In [13]:
# https://huggingface.co/docs/transformers/main/chat_templating#model-training
# 70 % context-grounded, 30% pure knowledge
import random
from datasets import Dataset

def create_chat_dataset(df, tokenizer):

    # context grounded (70%)
    context_grounded = df
    chat_list = context_grounded.apply(
        lambda x: format_row(x, tokenizer, include_context=True),axis=1
        ).to_list()

    # questions only (30%)
    questions_only = df.sample(n = int(len(df)*0.3/0.7))
    chat_list.extend(
        questions_only.apply(
            lambda x: format_row(x, tokenizer, include_context=False), axis=1
            )
        )

    # dataset
    dataset = Dataset.from_dict({"chat": chat_list})    

    return dataset


In [14]:
dataset = create_chat_dataset(df2, tokenizer)

In [15]:
# split into train, val and test
split_1 = dataset.train_test_split(test_size=0.2, seed = 42)
split_2 = split_1["test"]. train_test_split(test_size=0.5, seed = 42)

dataset_splits = {
    "train": split_1["train"],
    "val": split_2["train"],
    "test": split_2["test"]
}


In [16]:
dataset_splits

{'train': Dataset({
     features: ['chat'],
     num_rows: 8000
 }),
 'val': Dataset({
     features: ['chat'],
     num_rows: 1000
 }),
 'test': Dataset({
     features: ['chat'],
     num_rows: 1000
 })}

## Parameter Efficient Fine Tuning

In [31]:
from dotenv import load_dotenv
load_dotenv()

True

In [15]:
import wandb
wandb.login()

wandb.init(
    project="finance-bro",
    name="qlora-mistral-7b-run1",
    config={
        "model": "mistral-7b-instruct-v0.2",
        "lora_r": 16,
        "lora_alpha": 32,
        "target_modules": "all_7",
        "dataset_mix": "70ctx_30noctx",
        "max_seq_length": 2048,
        "quantization": "nf4_4bit"
    }
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: coffeedrunk to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [16]:
# Quantized 
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

# QLoRA config: 4-bit storage, BF16 compute
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4", # NF4 is better than INT4 for LLMs
    bnb_4bit_compute_dtype=torch.bfloat16,  # RTX 5070 Ti supports BF16 natively
    bnb_4bit_use_double_quant= True # quantize the quantization constants too → saves ~0.4GB extra

)

# trained_weights = "mistralai/Mistral-7B-Instruct-v0.2"

# quantized base model
base_model = AutoModelForCausalLM.from_pretrained(
    trained_weights,
    quantization_config=bnb_config,
    device_map = "auto")


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [17]:
# https://huggingface.co/docs/transformers/main/peft
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from transformers import AutoTokenizer, MistralForCausalLM


lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM, # type of task to train on
    r = 16, # dimension of the smallest matrix/rank adapter capacity
    lora_alpha = 32, # scaling factor
    lora_dropout= 0.05, # dropout of LoRA layers
    target_modules=["q_proj", "v_proj"],  # which layers get LoRA
    bias = "none", # freeze the bias terms (only train LoRA matrices)
     
)

# prepare model for 4-bit quantization (enables gradient checkpointing and casts the right layers to trainable precision)
base_model = prepare_model_for_kbit_training(base_model) 

# Adapt model for the Lora 
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()


trainable params: 6,815,744 || all params: 7,248,547,840 || trainable%: 0.0940


In [23]:
from trl import SFTTrainer , SFTConfig

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"   # required for causal LM training

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset_splits["train"],
    eval_dataset=dataset_splits["val"],
    processing_class=tokenizer,
    args=SFTConfig(
        dataset_text_field="chat",
        max_length=2048,
        report_to="wandb",
        output_dir="./finance-bro-checkpoints",
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        num_train_epochs=3,
        eval_strategy="steps",
        eval_steps=100,
        save_strategy="steps",
        save_steps=100,
        load_best_model_at_end=True,
        bf16=True,
    )
)


Adding EOS to train dataset:   0%|          | 0/8000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/8000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/8000 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
trainer.train()

trainer.model.save_pretrained("./finance-bro-adapter")
tokenizer.save_pretrained("./finance-bro-adapter")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss,Validation Loss
100,1.407469,1.483248
200,1.413453,1.332529
300,1.126477,1.246225
400,1.141202,1.221710
500,1.232786,1.212054
600,1.140999,1.203100
700,1.174698,1.196162
800,1.213567,1.191754
900,1.316389,1.186876
1000,1.157556,1.182429


TrainOutput(global_step=6000, training_loss=1.160076498190562, metrics={'train_runtime': 6038.8599, 'train_samples_per_second': 3.974, 'train_steps_per_second': 0.994, 'total_flos': 1.7378518693281792e+17, 'train_loss': 1.160076498190562})

In [ ]:
# trainer.model.save_pretrained("./finance-bro-adapter")
# tokenizer.save_pretrained("./finance-bro-adapter")

In [ ]:
# # Quick perplexity eval on test split
# test_results = trainer.evaluate(dataset_splits["test"])
# print(test_results)  # logs eval_loss on unseen data
# wandb.log({"test/final_loss": test_results["eval_loss"]})

# wandb.finish()


## Reload from checkpoints (and save)

In [1]:
# Load from checkpoints
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch

# Step 1 — reload base model with quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.2",
    quantization_config=bnb_config,
    device_map={"": 0}       # ← use this instead of "auto" — avoids the accelerate bug
)

tokenizer = AutoTokenizer.from_pretrained(
    "./finance-bro-checkpoints/checkpoint-6000"
)

# Step 2 — load the LoRA adapter on top
model = PeftModel.from_pretrained(
    base_model,
    "./finance-bro-checkpoints/checkpoint-6000"
)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [ ]:
# Save adapter only
model.save_pretrained("./finance-bro-adapter")
tokenizer.save_pretrained("./finance-bro-adapter")

# Merge and save full model (need unquantized base for this)
base_model_fp16 = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.2",
    torch_dtype=torch.bfloat16,
    device_map={"": 0}
)

('./finance-bro-adapter/tokenizer_config.json',
 './finance-bro-adapter/chat_template.jinja',
 './finance-bro-adapter/tokenizer.json')

In [ ]:
# After training, swap eval_dataset and call evaluate() once
trainer.eval_dataset = dataset_splits["test"]   # ← swap in place
test_results = trainer.evaluate()
wandb.log({"test/final_loss": test_results["eval_loss"]})

In [ ]:
# merged = model.merge_and_unload()
# merged.save_pretrained("./finance-bro-final")
# tokenizer.save_pretrained("./finance-bro-final")

/var/home/anne/Documents/_Ironhack/alpha-crunch/.venv/lib/python3.12/site-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./finance-bro-final/tokenizer_config.json',
 './finance-bro-final/chat_template.jinja',
 './finance-bro-final/tokenizer.json')

## Evaluate

In [19]:
# Rebuild trainer to evaluate
from trl import SFTTrainer, SFTConfig
import wandb

wandb.init(project="finance-bro", name="final-test-evaluation")

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset_splits["train"],
    eval_dataset=dataset_splits["test"], 
    processing_class=tokenizer,
    args=SFTConfig(
        output_dir="./finance-bro-checkpoints",
        dataset_text_field="chat",
        max_length=2048,
        bf16=True,
        report_to="wandb"
    )
)


# Run evaluation only — no training
test_results = trainer.evaluate()
print(test_results)
wandb.log({"test/final_loss": test_results["eval_loss"]})

wandb.finish()


Adding EOS to train dataset:   0%|          | 0/8000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/8000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/8000 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

{'eval_loss': 2.0017647743225098, 'eval_model_preparation_time': 0.0047, 'eval_runtime': 43.6619, 'eval_samples_per_second': 22.903, 'eval_steps_per_second': 2.863}


eval/entropy,▁
eval/loss,▁
eval/mean_token_accuracy,▁
eval/model_preparation_time,▁
eval/num_tokens,▁
eval/runtime,▁
eval/samples_per_second,▁
eval/steps_per_second,▁
test/final_loss,▁
train/global_step,▁▁
eval/entropy,1.24992


### LLM-as-Judge 

In [37]:
from openai import OpenAI
client = OpenAI()

def evaluate_finance_bro(question, context, reference_answer, model_answer):
    prompt = f"""You are evaluating a financial AI assistant. Score the answer 1-5.

Question: {question}
Context: {context}
Reference Answer: {reference_answer}
Model Answer: {model_answer}

Score on:
- Factual accuracy (is it financially correct?)
- Reasoning quality (does it think like a trader/analyst?)
- Completeness

Return JSON: {{"accuracy": X, "reasoning": X, "completeness": X, "overall": X}}"""
    
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content


In [36]:
dataset_splits["test"]["chat"][100]

'<s> [INST] Use the following context to answer the question.\n\n    Context: The information called for by Item 3 on Legal proceedings is incorporated by reference to the information set forth in Note 19 of the Notes to Consolidated Financial Statements in Item 8 of the report.\n\n    Question: How is the information for Item 3 on Legal proceedings disclosed in this report?\n     [/INST] The information for Item 3 on Legal proceedings is disclosed by referencing Note 19 in the Notes to Consolidated Financial Statements included in Item 8.</s>'

In [39]:
len(dataset_splits["test"]["chat"])

1000

In [38]:
import re

def parse_chat_sample(formatted_string):
    # Extract content between [INST] and [/INST] as the instruction
    inst_match = re.search(r'\[INST\](.*?)\[/INST\]', formatted_string, re.DOTALL)
    # Extract content after [/INST] as the answer
    answer_match = re.search(r'\[/INST\](.*?)</s>', formatted_string, re.DOTALL)
    
    instruction = inst_match.group(1).strip() if inst_match else ""
    answer = answer_match.group(1).strip() if answer_match else ""
    
    # Extract context and question from instruction
    if "Context:" in instruction and "Question:" in instruction:
        context = re.search(r'Context:(.*?)Question:', instruction, re.DOTALL)
        question = re.search(r'Question:(.*?)$', instruction, re.DOTALL)
        context = context.group(1).strip() if context else None
        question = question.group(1).strip() if question else instruction
    else:
        context = None
        question = instruction.strip()
    
    return {"question": question, "context": context, "answer": answer}

# Test it
sample = parse_chat_sample(dataset_splits["test"]["chat"][100])
print(sample["question"])
print(sample["context"])
print(sample["answer"])

How is the information for Item 3 on Legal proceedings disclosed in this report?
The information called for by Item 3 on Legal proceedings is incorporated by reference to the information set forth in Note 19 of the Notes to Consolidated Financial Statements in Item 8 of the report.
The information for Item 3 on Legal proceedings is disclosed by referencing Note 19 in the Notes to Consolidated Financial Statements included in Item 8.


In [40]:
def ask_finance_bro(question, context=None):
    if context:
        content = f"Use the following context to answer the question.\n\nContext: {context}\n\nQuestion: {question}"
    else:
        content = question

    chat = [{"role": "user", "content": content}]


    prompt = tokenizer.apply_chat_template(
        chat,
        tokenize=False,
        add_generation_prompt=True   # ← True for inference, adds trailing [/INST]
    )

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    # Strip the input prompt from output — decode only new tokens
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

print(sample["question"])
print(sample["context"])
print(sample["answer"])

model_answer = ask_finance_bro(question=sample["question"], context=sample["context"])
    
scores = evaluate_finance_bro(
    sample["question"], sample["context"], sample["answer"], model_answer
)

How is the information for Item 3 on Legal proceedings disclosed in this report?
The information called for by Item 3 on Legal proceedings is incorporated by reference to the information set forth in Note 19 of the Notes to Consolidated Financial Statements in Item 8 of the report.
The information for Item 3 on Legal proceedings is disclosed by referencing Note 19 in the Notes to Consolidated Financial Statements included in Item 8.


In [41]:
scores

'{"accuracy": 5, "reasoning": 5, "completeness": 5, "overall": 5}'

In [ ]:

# # Run on test split
# for row in dataset_splits["test"]["chat"]:
#     model_answer = generate(row["question"], row["context"])  # your Finance Bro
    
#     scores = evaluate_finance_bro(
#         row["question"], row["context"], row["answer"], model_answer
#     )
#     # log to W&B
#     wandb.log({"test/llm_judge": scores})


## Inference

In [22]:
sample = dataset_splits["test"][0]
print(sample)

{'chat': '<s> [INST] Use the following context to answer the question.\n\n    Context: Net revenue for fiscal year 2023 increased by $435 million compared to fiscal year 2022.\n\n    Question: How did the net revenue for fiscal year 2023 compare to fiscal year 2022?\n     [/INST] The net revenue for fiscal year 2023 increased by $435 million compared to fiscal year 2022.</s>'}


In [25]:
content = "Use the following context to answer the question.\n\n    Context: Net revenue for fiscal year 2023 increased by $435 million compared to fiscal year 2022.\n\n    Question: How did the net revenue for fiscal year 2023 compare to fiscal year 2022?\n"

chat = [{"role": "user", "content": content}]


prompt = tokenizer.apply_chat_template(
    chat,
    tokenize=False,
    add_generation_prompt=True   # ← True for inference, adds trailing [/INST]
)


In [32]:
prompt

'<s> [INST] Use the following context to answer the question.\n\n    Context: Net revenue for fiscal year 2023 increased by $435 million compared to fiscal year 2022.\n\n    Question: How did the net revenue for fiscal year 2023 compare to fiscal year 2022?\n [/INST]'

In [29]:
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

# Strip the input prompt from output — decode only new tokens
new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
answer = tokenizer.decode(new_tokens, skip_special_tokens=True)
print(answer)

# Test it
# print(ask_finance_bro("What does a high Fear & Greed score mean for S&P500 positioning?"))


The net revenue for fiscal year 2023 increased by $435 million compared to fiscal year 2022.


In [30]:
model.eval()

def ask_finance_bro(question, context=None):
    if context:
        content = f"Use the following context to answer the question.\n\nContext: {context}\n\nQuestion: {question}"
    else:
        content = question

    chat = [{"role": "user", "content": content}]


    prompt = tokenizer.apply_chat_template(
        chat,
        tokenize=False,
        add_generation_prompt=True   # ← True for inference, adds trailing [/INST]
    )

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    # Strip the input prompt from output — decode only new tokens
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

# Test it
print(ask_finance_bro("What does a high Fear & Greed score mean for S&P500 positioning?"))


A high Fear & Greed score for the S&P 500 index indicates that the market sentiment is leaning towards fear or extreme fear. This suggests that investors are more likely to be selling stocks, causing the market to become oversold. Conversely, a low Fear & Greed score indicates a bullish sentiment, where investors are more likely to be buying stocks, causing the market to become overbought. High Fear & Greed scores can signal potential buying opportunities for long-term investors, while low scores may indicate selling opportunities. However, it's important to note that the Fear & Greed index is just one tool used in analyzing market sentiment and should be considered in conjunction with other technical and fundamental analysis.
